# GELU — Gaussian Error Linear Units (2016)

_Papers · Foundations & Optimization_

**A smooth activation that weights each input by the probability a standard-normal draw falls below it — x times the normal CDF — instead of hard-gating by sign like ReLU.**

---

This notebook is a practice scaffold for the **GELU — Gaussian Error Linear Units (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# ---- GELU from scratch: EXACT erf form (Section 2 of Hendrycks & Gimpel, 2016) ----
def my_gelu_exact(x):
    Phi = 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))   # standard-normal CDF
    return x * Phi                                        # GELU(x) = x * Phi(x)

# ---- GELU tanh approximation (Section 2) ----
def my_gelu_tanh(x):
    c = math.sqrt(2.0 / math.pi)                          # ~0.7979
    return 0.5 * x * (1.0 + torch.tanh(c * (x + 0.044715 * x**3)))

# ---- THE ORACLE: exact form must equal F.gelu (PyTorch's default) ----
x = torch.linspace(-4, 4, 1001)
print("allclose(exact, F.gelu):",
      torch.allclose(my_gelu_exact(x), F.gelu(x), atol=1e-6))            # expect True
print("max abs diff vs F.gelu (exact):",
      (my_gelu_exact(x) - F.gelu(x)).abs().max().item())                 # ~2e-7

# tanh approx: close to exact, and matches PyTorch's approximate='tanh'
print("max abs diff (tanh approx vs exact F.gelu):",
      (my_gelu_tanh(x) - F.gelu(x)).abs().max().item())                  # small, ~1e-3
print("allclose(my tanh, F.gelu approximate='tanh'):",
      torch.allclose(my_gelu_tanh(x), F.gelu(x, approximate='tanh'), atol=1e-6))  # True

# ---- recompute the worked example: x = 1, -1, 0 ----
for xv in [1.0, -1.0, 0.0]:
    t = torch.tensor([xv])
    print(f"GELU({xv:+.0f}) = {my_gelu_exact(t).item():.4f}  "
          f"(ReLU = {torch.relu(t).item():.4f})")
# expect: GELU(+1)=0.8413, GELU(-1)=-0.1587, GELU(0)=0.0000

# ---- ABLATION: same tiny MLP, swap GELU <-> ReLU on a toy 2-class problem ----
def make_net(act):
    return nn.Sequential(nn.Linear(2, 16), act, nn.Linear(16, 16), act, nn.Linear(16, 2))

# toy two-cluster classification
torch.manual_seed(0)
N = 200
Xa = torch.randn(N, 2) + torch.tensor([ 1.5,  1.5])
Xb = torch.randn(N, 2) + torch.tensor([-1.5, -1.5])
Xd = torch.cat([Xa, Xb]); yd = torch.cat([torch.zeros(N), torch.ones(N)]).long()

def train(act, steps=200):
    torch.manual_seed(1)                                  # same init for both
    net = make_net(act); opt = torch.optim.Adam(net.parameters(), lr=0.05)
    for _ in range(steps):
        opt.zero_grad(); loss = F.cross_entropy(net(Xd), yd); loss.backward(); opt.step()
    acc = (net(Xd).argmax(1) == yd).float().mean().item()
    return loss.item(), acc

g_loss, g_acc = train(nn.GELU())
r_loss, r_acc = train(nn.ReLU())
print(f"GELU: final loss {g_loss:.4f}, acc {g_acc:.3f}")
print(f"ReLU: final loss {r_loss:.4f}, acc {r_acc:.3f}")

## Visualize the data & results

_What do GELU and ReLU look like side by side, and does swapping ReLU->GELU help a tiny MLP on a toy 2-class problem (everything else held fixed)?_

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F

# LEFT chart: activation curves
def gelu(x): return x * 0.5 * (1 + torch.erf(x / math.sqrt(2)))
xs = torch.arange(-4, 4.01, 0.25)
print("x, GELU, ReLU:")
for x in xs:
    print(round(x.item(),2), round(gelu(x).item(),4), round(torch.relu(x).item(),4))

# RIGHT chart: GELU vs ReLU on a toy 2-class MLP (everything else fixed)
torch.manual_seed(0)
N = 200
Xd = torch.cat([torch.randn(N,2)+torch.tensor([1.5,1.5]),
                torch.randn(N,2)+torch.tensor([-1.5,-1.5])])
yd = torch.cat([torch.zeros(N), torch.ones(N)]).long()

def train(act, steps=200):
    torch.manual_seed(1)                          # identical initialization
    net = nn.Sequential(nn.Linear(2,16), act, nn.Linear(16,16), act, nn.Linear(16,2))
    opt = torch.optim.Adam(net.parameters(), lr=0.05)
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy(net(Xd), yd).backward(); opt.step()
    return (net(Xd).argmax(1)==yd).float().mean().item()

print("GELU acc:", round(train(nn.GELU()),4))     # ~0.9950
print("ReLU acc:", round(train(nn.ReLU()),4))     # ~0.9950

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute $\mathrm{GELU}(2)$ exactly, given $\operatorname{erf}(2/\sqrt2)=\operatorname{erf}(1.4142)\approx0.9545$. How does it compare to $\mathrm{ReLU}(2)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scale the input: $z=2/\sqrt2=1.4142$. — _The erf takes $x/\sqrt2$, not $x$._
- Normal CDF: $\Phi(2)=\tfrac12(1+0.9545)=0.9773$. — _$\Phi(x)=\tfrac12[1+\operatorname{erf}(x/\sqrt2)]$._
- GELU: $2\times0.9773=1.9545$. — _Weight the input by the probability._

**Answer:** $\mathrm{GELU}(2)\approx1.9545$, slightly below $\mathrm{ReLU}(2)=2$. For large positive inputs $\Phi(x)\to1$, so GELU approaches ReLU from below — it passes big positives through almost unchanged.

</details>

**Problem 2.** Without computing erf, argue what $\mathrm{GELU}(-3)$ is approximately, and why ReLU and GELU nearly agree there.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note $\Phi(-3)$ is the probability a standard normal is $\le-3$, which is about $0.0013$. — _Three standard deviations below the mean is very unlikely._
- GELU: $(-3)\times0.0013\approx-0.004$. — _Weight the input by that tiny probability._
- ReLU$(-3)=0$. — _Hard gate on the negative sign._

**Answer:** $\mathrm{GELU}(-3)\approx-0.004$, almost exactly ReLU's $0$. For strongly negative inputs $\Phi(x)\to0$, so GELU crushes them toward zero just like ReLU — the two functions agree in the tails and differ mainly in the soft knee near the origin.

</details>

**Problem 3.** Ablation: in the CODE, train the same tiny MLP on a toy 2-class problem twice — once with GELU, once with ReLU — and compare final loss/accuracy. Then explain what the swap changed and what it did not.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build one small network; swap only the activation between $\mathrm{GELU}$ and $\mathrm{ReLU}$, keeping seed, data, width and optimizer fixed. — _Isolate the activation as the only variable._
- Train both for the same number of steps and record final loss and accuracy. — _A fair head-to-head on identical conditions._
- Inspect the loss curves: GELU's smooth gradient near 0 vs ReLU's hard kink. — _Smoothness is the qualitative claim being tested._

**Answer:** In our small run GELU and ReLU tie on accuracy (both ~0.9950) with very close final losses (see CODEVIZ — our small run, not the paper's number); the task is easy enough that both saturate. The swap only changed the nonlinearity; the architecture, data, and optimizer are identical. This mirrors the paper's qualitative finding that GELU matches ReLU on simple tasks. The effect is small on a toy task — GELU's advantage is most pronounced in the large Transformers it later became standard in.

</details>